In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("housing.csv")

In [6]:
df["income_cat"] = pd.cut(df["median_income"] , bins = [0,1.5,3.0,4.5,6.0,np.inf] ,
                          labels=[1,2,3,4,5])

In [7]:
from sklearn.model_selection import StratifiedShuffleSplit
split = StratifiedShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
for train_index , test_index in split.split(df , df["income_cat"]):
    strat_train_set = df.loc[train_index]  # We will use train set for practice , and for execution we will use test set
    strat_test_set = df.loc[test_index]

In [8]:
# Remove the income cat cols

for sett in (strat_test_set,strat_train_set):
    sett.drop("income_cat" , axis=1 , inplace=True)

In [9]:
df = strat_train_set.copy()

In [10]:
housing = strat_train_set.drop("median_house_value" , axis=1)

housing = housing.drop("ocean_proximity" ,axis=1)
housing

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income
12655,-121.46,38.52,29.0,3873.0,797.0,2237.0,706.0,2.1736
15502,-117.23,33.09,7.0,5320.0,855.0,2015.0,768.0,6.3373
2908,-119.04,35.37,44.0,1618.0,310.0,667.0,300.0,2.8750
14053,-117.13,32.75,24.0,1877.0,519.0,898.0,483.0,2.2264
20496,-118.70,34.28,27.0,3536.0,646.0,1837.0,580.0,4.4964
...,...,...,...,...,...,...,...,...
15174,-117.07,33.03,14.0,6665.0,1231.0,2026.0,1001.0,5.0900
12661,-121.42,38.51,15.0,7901.0,1422.0,4769.0,1418.0,2.8139
19263,-122.72,38.44,48.0,707.0,166.0,458.0,172.0,3.1797
19140,-122.70,38.31,14.0,3155.0,580.0,1208.0,501.0,4.1964


In [11]:
# Consructing an pipeline

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [12]:
pipeline = Pipeline([
    ("impute" , SimpleImputer(strategy="median")) ,
    ("standardize" , StandardScaler()),
])

In [13]:
pipeline.fit_transform(housing)

array([[-0.94135046,  1.34743822,  0.02756357, ...,  0.73260236,
         0.55628602, -0.8936472 ],
       [ 1.17178212, -1.19243966, -1.72201763, ...,  0.53361152,
         0.72131799,  1.292168  ],
       [ 0.26758118, -0.1259716 ,  1.22045984, ..., -0.67467519,
        -0.52440722, -0.52543365],
       ...,
       [-1.5707942 ,  1.31001828,  1.53856552, ..., -0.86201341,
        -0.86511838, -0.36547546],
       [-1.56080303,  1.2492109 , -1.1653327 , ..., -0.18974707,
         0.01061579,  0.16826095],
       [-1.28105026,  2.02567448, -0.13148926, ..., -0.71232211,
        -0.79857323, -0.390569  ]], shape=(16512, 8))

In [14]:
# Column transformer --> final preprocessing

In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score

In [16]:
# 1.load the dataset
housing = pd.read_csv("housing.csv")

In [17]:
# 2. Create a Stratified test set
housing["income_cat"] = pd.cut(housing["median_income"] , 
                               bins=[0,1.5,3.0,4.5,6.0,np.inf] ,
                              labels=[1,2,3,4,5])

splits = StratifiedShuffleSplit(n_splits=1 , test_size=0.2 ,random_state=42)

for test_index,train_index in splits.split(housing , housing["income_cat"]):
    strat_train_set = housing.loc[train_index].drop("income_cat" , axis=1)
    strat_test_set = housing.loc[test_index].drop("income_cat" , axis=1)

In [18]:
# Work on a training data
housing = strat_train_set.copy()

In [19]:
# 3. Separate predictors/features and labels
housing_labels = housing["median_house_value"].copy() # labels
housing = housing.drop("median_house_value", axis=1) # features

In [20]:
# 4. Separate numerical and categorical columns
num_attribs = housing.drop("ocean_proximity", axis=1).columns.tolist()
cat_attribs = ["ocean_proximity"]

In [21]:
# 5. Pipelines
# Numerical pipeline
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

In [22]:
# Categorical pipeline
cat_pipeline = Pipeline([
    # ("ordinal", OrdinalEncoder())  # Use this if you prefer ordinal encoding
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [23]:
 # Full pipeline -->column transformer
full_pipeline = ColumnTransformer([
    ("num", num_pipeline, num_attribs),
    ("cat", cat_pipeline, cat_attribs),
])

In [24]:
# 6. Transform the data
housing_prepared = full_pipeline.fit_transform(housing)

In [25]:
print(housing_prepared)

[[ 0.57507019 -0.69657252  0.0329564  ...  0.          0.
   0.        ]
 [-0.43480141 -0.33466769 -0.36298077 ...  0.          0.
   0.        ]
 [ 0.54522177 -0.63547171  0.58726843 ...  0.          0.
   0.        ]
 ...
 [-0.08656982 -0.54617051  1.14158047 ...  0.          0.
   0.        ]
 [ 0.81385757 -0.92687559  0.11214383 ...  0.          0.
   0.        ]
 [ 0.49049967 -0.66367208  0.58726843 ...  0.          0.
   0.        ]]


In [29]:
# 7 train the model
# linear regression model
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared , housing_labels)
lin_pred = lin_reg.predict(housing_prepared)
lin_rmse = root_mean_squared_error(housing_labels , lin_pred)
print(lin_rmse)

67272.19418925028


In [30]:
# Decision tree model --> overfitting
dec_reg = DecisionTreeRegressor()
dec_reg.fit(housing_prepared , housing_labels)
dec_pred = dec_reg.predict(housing_prepared)
dec_rmse = root_mean_squared_error(housing_labels , dec_pred)
print(dec_rmse)

0.0


In [31]:
# Random forest  model
rf_reg = RandomForestRegressor()
rf_reg.fit(housing_prepared , housing_labels)
rf_pred = rf_reg.predict(housing_prepared)
rf_rmse = root_mean_squared_error(housing_labels , rf_pred)
print(rf_rmse)

19951.540636253514


In [35]:
# Cross Validation
dec_reg = DecisionTreeRegressor()
dec_reg.fit(housing_prepared , housing_labels)
dec_pred = dec_reg.predict(housing_prepared)
tree_rmses = -cross_val_score(dec_reg ,
                              housing_prepared , 
                              housing_labels ,
                              scoring="neg_root_mean_squared_error" , 
                              cv=10)
print(pd.Series(tree_rmses).describe())

count       10.000000
mean     73656.105455
std       5385.430299
min      66502.302282
25%      71846.401698
50%      73003.787376
75%      73950.830053
max      85205.874964
dtype: float64
